# Capstone G2: Skin cancer screening

Seven kinds of skin spot from dermatoscope photos -- including melanoma, the dangerous one. This is a **screening** tool, so its #1 job is to *not miss* a cancer.

**Your group's priority: sensitivity (don't miss the dangerous class).** In screening, missing a real melanoma is far worse than a false alarm: a false alarm gets a second look, a miss goes home untreated. Overall accuracy can look great while the model quietly misses the rare, deadly class.

**Your goal (the rubric):** build it, check honestly how well it works, find a case where it fails, defend one choice you made, and make sure *both* partners can explain everything.

### How to use this notebook (read this first)

- A notebook is a stack of **cells**. A gray cell is **code**; this white cell is a note.
- To run a code cell, click it and press **Shift+Enter**. Run them **top to bottom, in order** --
  later cells use things earlier cells made.
- **Red text is normal**, not a disaster -- it's usually a warning. A real error stops with a
  message; read the last line, or paste it to Claude and ask "what does this mean?"
- You will not understand every line, and that's fine. But you should be able to say, in plain
  English, *what each cell is for*. Stuck on a line? Ask Claude: **"explain this cell line by line."**
- Nothing you click here can break anything. Experiment.

### You have Claude Code -- so here's your *real* job

You can already make a computer write code: just describe what you want to Claude Code and it will
build it. So **writing code is no longer the hard part.** The skill that actually makes you good at
this -- the thing this notebook is training -- is **judgment**:

1. **What are you trying to build?** "Accurate" isn't a goal. A cancer screen that must never miss a
   tumor is a *different tool* than one that's just right-on-average. Decide what "good" means first.
2. **What tools exist?** Different models, different ways to prep the data, different ways to grade a
   model. This notebook is a guided tour of that toolbox.
3. **Why does each tool exist** -- what problem it was invented to solve.
4. **How do you pick the right one?** That's the whole game. The wrong tool gives a wrong answer even
   with flawless code.

So: read the notes, play with the dials, and when you want to change or add something, **describe it
to Claude Code and let it write the code** -- then be ready to explain *why you chose it*. That "why"
is what you're graded on, and it's what a real scientist or doctor would ask you.

In [ ]:
# WHAT THIS DOES: gets the course files onto this Colab machine and installs the tools we need.
# (On your own computer it does nothing.) Just run it once and wait for "setup ok".
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g2_skin_screening")
sys.path.insert(0, "../..")            # so we can import the course helper files
sys.path.insert(0, "../../../_shared")
import colab_setup
colab_setup.ensure(*colab_setup.DAY1, "medmnist")

### First, some plain-English vocabulary

- A **model** is a rule the computer writes *by itself* after seeing lots of examples -- like a kid
  who learns "cat vs dog" from photos, not from a dictionary definition.
- **Training** = showing it labeled examples so it can find that rule.
- To a computer, an **image is just a grid of numbers** (how bright each pixel is). The model does
  math on those numbers. It never "sees" a picture the way you do.
- We split our data into a **training set** (the model studies these) and a **test set** (held back,
  like a real exam) -- because a model that just *memorized* the study set would ace the homework and
  flunk the exam. The **test score is the only one that counts.**

In [ ]:
# WHAT THIS DOES: loads the images, splits them into train/test, and tells us the class names.
import torch, sys
sys.path.insert(0, "../..")
import capstone_common as cc

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)   # 'cuda' = a GPU is helping (fast). 'cpu' = slower but still fine.

FLAG = "dermamnist"
train_loader, val_loader, test_loader, n_classes, task = cc.get_loaders(FLAG, size=64)
CLASS_NAMES = cc.class_names(FLAG)
print("classes:", n_classes, "->", CLASS_NAMES, "| task:", task)

### Look at the data first
**What this does:** draws a few images with their labels. Always eyeball your data before modeling -- can *you* even tell the classes apart?

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))   # grab one batch of images
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, im, lb in zip(axes, imgs, labels):
    ax.imshow(im[0], cmap="gray")
    ax.set_title(CLASS_NAMES[int(lb)], fontsize=8); ax.axis("off")
plt.tight_layout()

### Build a first model (the baseline)

**What this does:** trains a model in a few seconds using a shortcut called **transfer learning**,
then prints its **test accuracy** (the fraction it got right on images it never studied).

**Why this works -- the analogy:** instead of teaching a computer to see *from scratch* (which needs
millions of images), we **borrow a network that already learned to see** from millions of everyday
photos. It already knows edges, shapes, and textures. We keep all of that ("**freeze**" it) and only
train a tiny final decision layer (a new "**head**") for our specific job. It's like hiring someone
who already speaks the language and just teaching them the medical words.

In [ ]:
model = cc.make_baseline(n_classes)                                   # borrowed "eyes" + a fresh decision layer
model = cc.train(model, train_loader, val_loader, epochs=3, device=device)  # 3 passes over the data
test_acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
print(f"\nbaseline TEST accuracy: {test_acc:.3f}")   # e.g. 0.800 = right 80% of the time

### Now build your OWN model (interactive -- no coding needed)

**What this does:** gives you dials. The dials come **pre-set to a strong recipe** -- just press
**"Run Interact"** once and you should already get a good number (give it a minute or two to train).
*Then* change **one dial at a time** to learn what actually matters. Write every result down -- that
log is half your presentation.

**What each dial means, in plain terms:**
- **backbone** -- which "brain" to borrow. `resnet18` is small and fast; the others are bigger
  (sometimes smarter, sometimes just slower).
- **pretrained** -- ON = start from the brain that already learned to see (usually much better).
  OFF = start from a *blank* brain. With only a few thousand images, blank is like learning to read
  and diagnose at the same time -- too much at once, so it does worse. **Toggle it to see the drop.**
- **unfreeze** -- also re-train the borrowed brain, not just the head. This is what pushes results
  from "okay" to "great" -- but only if you pair it with a **small learning rate** (see below).
- **augment** -- show each image flipped/rotated. Teaches "the lesion," not "the lesion is top-left."
- **learn rate** -- how big a step the model takes each update. **The #1 reason results disappoint:**
  when you **unfreeze**, use a *small* rate like **0.0001** (0.001 is too big and wrecks the borrowed
  brain). When the backbone is **frozen**, **0.001** is fine.
- **epochs** -- how many times it studies the whole set (you can go up to 100). More usually helps
  when unfrozen, up to a point -- then it starts memorizing (overfitting).

**The recipe that works (already the default):** pretrained ON, unfreeze ON, augment ON,
learn rate **0.0001**, **~15+ epochs**. That combination -- fine-tuning a pretrained model, gently,
for long enough -- is how you get results that actually land. If yours "can't come close," it's
almost always: froze the backbone, or used too big a learning rate, or trained too few epochs.

In [ ]:
from ipywidgets import interact_manual, Dropdown, Checkbox, IntSlider

# The defaults below are ALREADY a strong recipe (pretrained + unfrozen + augmented + a low learning
# rate + enough epochs). Just press "Run Interact" and you should get a good number -- THEN start
# experimenting one dial at a time to understand why it works.
def build_model(backbone="resnet18", pretrained=True, unfreeze_backbone=True, augment=True,
                learning_rate=1e-4, epochs=15):
    global model, train_loader, val_loader, test_loader
    train_loader, val_loader, test_loader, n_cls, _ = cc.get_loaders(FLAG, size=64, augment=augment)
    model = cc.make_model(n_cls, backbone=backbone, pretrained=pretrained, unfreeze_backbone=unfreeze_backbone)
    model = cc.train(model, train_loader, val_loader, epochs=epochs, lr=learning_rate, device=device)
    acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
    print(f"\n>>> TEST accuracy = {acc:.3f}   [{backbone}, pretrained={pretrained}, "
          f"unfreeze={unfreeze_backbone}, augment={augment}, lr={learning_rate}, {epochs}ep]")

try:
    interact_manual(build_model,
        backbone=Dropdown(options=cc.BACKBONES, value="resnet18", description="backbone"),
        pretrained=Checkbox(value=True, description="pretrained"),
        unfreeze_backbone=Checkbox(value=True, description="unfreeze"),
        augment=Checkbox(value=True, description="augment"),
        learning_rate=Dropdown(options=[("0.0001  (best when UNFROZEN)", 1e-4), ("0.0003", 3e-4),
                                        ("0.001  (best when frozen)", 1e-3), ("0.003", 3e-3)],
                               value=1e-4, description="learn rate", style={"description_width": "initial"}),
        epochs=IntSlider(value=15, min=1, max=100, step=1, description="epochs"))
except ImportError:
    build_model()

### Grade your model like a regulator

A high accuracy is **not** enough for medicine. Yesterday you set the *rules*; now hold **your**
model to them. Pick a tool below and run it. Your group's priority is **sensitivity (don't miss the dangerous class)**, so the
menu starts on **Failure analysis** -- but try the others too.

**What each tool tells you (in plain terms):**
- **Failure analysis** -- shows the melanoma cases it got wrong. For screening, that's the whole ballgame.
- **Fairness** -- is accuracy even across all 7 spot types, or does it flunk the rare ones?
- **Confusion matrix** -- a scorecard of truth vs guess. **Grad-CAM** -- a heat-map of where it looked. **Monitoring** -- does it survive blurrier photos?

**How to choose which audit:** start from *what would hurt a patient most* in your job. Screening
where a miss is deadly? Lead with **failure analysis**. Worried the model treats groups unequally?
**Fairness**. Not sure it's even looking at the right thing? **Grad-CAM**. The priority drives the tool.

*(Train a model first -- run the baseline or the builder above.)*

In [ ]:
from ipywidgets import interact_manual, Dropdown

TOOLS = list(cc.REGULATOR_TOOLS)
DEFAULT = [t for t in TOOLS if t.startswith("Failure")][0]

def audit(tool):
    if "model" not in globals():
        print("Train a model first (run the baseline or the builder above)."); return
    cc.REGULATOR_TOOLS[tool](model, test_loader, device=device, class_names=CLASS_NAMES)

try:
    interact_manual(audit, tool=Dropdown(options=TOOLS, value=DEFAULT,
                                          description="priority", style={"description_width": "initial"}))
except ImportError:
    audit(DEFAULT)

### One strong approach (peek only if you're stuck)

*This is roughly what the worked solution does. It's a nudge, not the code -- describe these steps to
Claude and build them yourself, then be ready to explain why.*

Screening cares about **sensitivity** (catching every melanoma) far more than raw accuracy. So after you train, don't just accept the default 0.5 cutoff -- **lower the decision threshold** so the model flags more suspicious spots, catching more melanomas even at the cost of a few extra false alarms. Report melanoma **recall before vs after** tuning. Also try a **stronger backbone** than a small ResNet, and check with **Grad-CAM** that the model is looking at the lesion, not the background.

### Where to take it (ask Claude to help)
- Which class does it miss most? Try **class weighting** or **augment** to fix it, and prove the fix on the confusion matrix.
- Swap in the full HAM10000 set (Kaggle), then feed it a phone photo of a mole -- watch it stumble on data that looks nothing like training.